# 02 – 資料探索（互動式）

本 Notebook 對已建置的各層資料進行互動式視覺化，涵蓋：

| 章節 | 內容 |
|------|------|
| 1 | 資料建置報告與站點維度總覽 |
| 2 | 事故樣本的時間與路線分布 |
| 3 | 時空矩陣覆蓋率分析 |
| 4 | 互動式事故時空速度熱圖 |
| 5 | 觀測速度 vs. 歷史基準對比 |

> **前置條件**：已執行 `build_station_vd_drive.py` 與 `build_accident_samples.py`，
> 且 `data/` 目錄下各 parquet 檔案完整存在。

In [ ]:
import os, sys, json
from pathlib import Path

# 環境偵測：Colab 或本機
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    # Ensure Noto CJK is available for matplotlib Chinese rendering
    import subprocess
    subprocess.run(
        ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
        check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    _DEFAULT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    _here = Path(os.getcwd()).resolve()
    for _p in [_here, *_here.parents]:
        if (_p / 'CLAUDE.md').exists():
            _DEFAULT_ROOT = str(_p)
            break
    else:
        _DEFAULT_ROOT = str(_here)

PROJECT_ROOT = os.environ.get('PROJECT_ROOT', _DEFAULT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
%cd {PROJECT_ROOT}

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as widgets
from IPython.display import display, clear_output

from scripts.lib.data_loaders import (
    duckdb_connection, load_samples, read_vd_dim, read_historical_profile,
)
from scripts.lib.calendar import day_class
from scripts.lib.visualization import heatmap, diff_heatmap, compare_heatmaps, configure_cjk_font
from scripts.lib.windows import rasterize_window

plt.rcParams['figure.dpi'] = 120
_cjk_font = configure_cjk_font()
print(f'CJK font     = {_cjk_font}')
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'IN_COLAB     = {IN_COLAB}')

## 1. 資料建置總覽

讀取 `data/metadata/station_build_report.json`（由 `build_station_vd_drive.py` 產生）
與 `data/metadata/vd_dim.parquet`，確認資料涵蓋日期範圍、站點數量與磁碟占用量。

In [ ]:
# 建置報告
with open(f'{PROJECT_ROOT}/data/metadata/station_build_report.json', encoding='utf-8') as _f:
    rpt = json.load(_f)

totals = rpt['totals']
rng    = rpt['range']

print('=' * 50)
print(f"資料範圍  ：{rng['start']} ～ {rng['end']}（{rng['days']} 天）")
print(f"原始資料  ：{totals['raw_rows']:>15,} 列")
print(f"station_1m：{totals['station_1m_rows']:>15,} 列  /  {totals['station_1m_size_mb']:>8,.0f} MB")
print(f"station_5m：{totals['station_5m_rows']:>15,} 列  /  {totals['station_5m_size_mb']:>8,.0f} MB")
print(f"建置狀態  ：{rpt['statuses']}")
print('=' * 50)

# VD 維度表統計
dim = read_vd_dim()
print(f"\n站點總數（vd_dim）：{len(dim):,}")
fw_summary = (
    dim.groupby('freeway')
    .agg(
        VD數量=('vdid', 'count'),
        最早資料=('first_source_date', 'min'),
        最新資料=('last_source_date', 'max'),
    )
    .reset_index()
)
display(fw_summary)

## 2. 事故樣本統計

利用 DuckDB 查詢事故樣本 metadata（`accident_samples_5m_metadata.parquet`），
分析各路線事故數量與逐日分布。

下拉選單可篩選特定國道，圖表即時更新。

In [ ]:
con = duckdb_connection()

by_fw = con.sql("""
    SELECT freeway, direction, COUNT(*) AS 事故數
    FROM accident_samples_metadata_5m
    GROUP BY freeway, direction
    ORDER BY freeway, direction
""").df()

by_day_all = con.sql("""
    SELECT CAST(report_time AS DATE) AS 日期, COUNT(*) AS 事故數
    FROM accident_samples_metadata_5m
    GROUP BY 1 ORDER BY 1
""").df()

# 篩選 widget
_fw_opts = ['全部'] + sorted(by_fw['freeway'].unique().tolist())
fw_sel = widgets.Dropdown(
    options=_fw_opts, value='全部',
    description='國道篩選：',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px'),
)

out_acc = widgets.Output()

def _refresh_accidents(change=None):
    fw = fw_sel.value
    mask = by_fw['freeway'] == fw if fw != '全部' else pd.Series([True] * len(by_fw))
    if fw == '全部':
        day_data = by_day_all
    else:
        day_data = con.sql(f"""
            SELECT CAST(report_time AS DATE) AS 日期, COUNT(*) AS 事故數
            FROM accident_samples_metadata_5m
            WHERE freeway = '{fw}'
            GROUP BY 1 ORDER BY 1
        """).df()

    with out_acc:
        clear_output(wait=True)
        display(by_fw[mask].reset_index(drop=True))
        fig, ax = plt.subplots(figsize=(13, 3))
        ax.bar(day_data['日期'], day_data['事故數'], width=1.2, color='steelblue', alpha=0.85)
        label = f'（{fw}）' if fw != '全部' else '（全部路線）'
        ax.set_title(f'每日事故通報數 {label}')
        ax.set_xlabel('日期')
        ax.set_ylabel('事故數')
        ax.xaxis.set_major_locator(mticker.MaxNLocator(12))
        fig.tight_layout()
        plt.show()

fw_sel.observe(_refresh_accidents, names='value')
display(fw_sel, out_acc)
_refresh_accidents()

## 3. 樣本矩陣覆蓋率

每個事故建立一個 **n\_space × n\_time** 的速度矩陣。
VD 布點稀疏或資料缺漏時，矩陣內會出現 NaN，
`filled_ratio` 代表有效格比例（1.0 = 完全填滿）。

拖曳滑桿可即時查看不同覆蓋率門檻下可用的事故數量。

In [ ]:
samples = load_samples()
meta    = samples.metadata.copy()
window  = samples.window

print(f'事故樣本總數 ：{len(meta):,}')
print(f'矩陣尺寸     ：{window.n_space} 空間格 × {window.n_time} 時間格')
print(f'  上游 {window.upstream_km} km | 下游 {window.downstream_km} km | '
      f'事前 {window.time_before_min} min | 事後 {window.time_after_min} min')

thresh_w = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description='覆蓋率門檻：',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False,
)

out_cov = widgets.Output()

def _refresh_cov(change=None):
    th = thresh_w.value
    passed = (meta['filled_ratio'] >= th).sum()
    with out_cov:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 3.5))
        ax.hist(meta['filled_ratio'], bins=40, color='steelblue', alpha=0.8, edgecolor='white')
        ax.axvline(th, color='crimson', linestyle='--', lw=2,
                   label=f'門檻 {th:.2f}　→　通過 {passed} / {len(meta)} 筆')
        ax.set_xlabel('filled_ratio（矩陣覆蓋率）')
        ax.set_ylabel('事故數')
        ax.set_title('事故樣本時空矩陣覆蓋率分布')
        ax.legend()
        fig.tight_layout()
        plt.show()

thresh_w.observe(_refresh_cov, names='value')
display(thresh_w, out_cov)
_refresh_cov()

## 4. 互動式樣本時空熱圖

從樣本中篩選事故，並以熱圖呈現時空速度場。

| 軸向 | 說明 |
|------|------|
| X 軸 | 距通報時刻的偏移（分鐘），**紅色虛線** = 通報時刻（offset = 0）|
| Y 軸 | 距事故點的上游距離（km），**黑色虛線** = 事故點（distance = 0）|
| 顏色 | 速度（km/h）—— 紅色偏低、綠色偏高 |

調整國道、方向、最低覆蓋率後，用「排名」滑桿逐一瀏覽符合條件的事故。

In [ ]:
_fws  = ['全部'] + sorted(meta['freeway'].unique().tolist())
_dirs = ['全部', 'N', 'S', 'E', 'W']

fw_w2  = widgets.Dropdown(
    options=_fws, value='全部', description='國道：',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='160px'),
)
dir_w2 = widgets.Dropdown(
    options=_dirs, value='全部', description='方向：',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='150px'),
)
cov_w2 = widgets.FloatSlider(
    value=0.6, min=0.0, max=1.0, step=0.05, description='最低覆蓋率：',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'),
    continuous_update=False,
)
rank_w = widgets.IntSlider(
    value=1, min=1, max=50, step=1, description='排名（覆蓋率↓）：',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='420px'),
)

out_heat = widgets.Output()
_cache = [meta.sort_values('filled_ratio', ascending=False).reset_index()]  # mutable

def _apply_filter(change=None):
    fw, dr, th = fw_w2.value, dir_w2.value, cov_w2.value
    mask = meta['filled_ratio'] >= th
    if fw != '全部':
        mask &= meta['freeway'] == fw
    if dr != '全部':
        mask &= meta['direction'] == dr
    filtered = meta[mask].sort_values('filled_ratio', ascending=False).reset_index()
    _cache[0] = filtered
    rank_w.max = max(len(filtered), 1)
    _show_sample()

def _show_sample(change=None):
    filtered = _cache[0]
    with out_heat:
        clear_output(wait=True)
        if filtered.empty:
            print('沒有符合條件的事故，請放寬篩選條件。')
            return
        r   = min(rank_w.value, len(filtered)) - 1
        row = filtered.iloc[r]
        idx = int(row['index'])
        obs = samples.matrix[idx]
        print(
            f"[{r+1}/{len(filtered)}]  accident_id={row['accident_id']}  "
            f"{row['freeway']}-{row['direction']}  "
            f"{row['mileage_km']}K  filled={row['filled_ratio']:.2f}"
        )
        heatmap(obs, window, row.to_dict(), title=f"觀測速度熱圖（{row['accident_id']}）")
        plt.show()

for _w in (fw_w2, dir_w2, cov_w2):
    _w.observe(_apply_filter, names='value')
rank_w.observe(_show_sample, names='value')

display(widgets.HBox([fw_w2, dir_w2]), cov_w2, rank_w, out_heat)
_apply_filter()

## 5. 觀測速度 vs. 歷史基準對比

對選定事故，從 `historical_speed_profile_5m/` 讀取相同路線的正常速度 profile，
截取對應的時空區間，與觀測值並列顯示差異。

| 圖 | 說明 |
|----|------|
| 左 | 觀測速度（實際發生情況）|
| 中 | 歷史基準速度（無事故時的期望值）|
| 右 | 差異（觀測 − 歷史）：藍色 = 速降、紅色 = 速升 |

> 若 `historical_speed_profile_5m/` 尚不存在，
> 請先執行 `uv run scripts/data/build_historical_profile.py` 再重跑本格。

In [ ]:
# 取覆蓋率最高的事故作為展示案例
top     = meta.sort_values('filled_ratio', ascending=False).iloc[50]
top_idx = int(meta.index[meta['accident_id'] == top['accident_id']][0])
obs_mat = samples.matrix[top_idx]

print(
    f"展示事故：{top['accident_id']}  "
    f"{top['freeway']}-{top['direction']}  "
    f"{top['mileage_km']}K  覆蓋率={top['filled_ratio']:.2f}"
)

PEAK_AM = (7 * 60, 9 * 60)
PEAK_PM = (17 * 60, 19 * 60)

def _peak_class(tod_min: int) -> str:
    if PEAK_AM[0] <= tod_min < PEAK_AM[1] or PEAK_PM[0] <= tod_min < PEAK_PM[1]:
        return 'peak'
    return 'nonpeak'

try:
    profile_df  = read_historical_profile(freeway=top['freeway'])
    report_time = pd.Timestamp(top['report_time'])
    direction   = top['direction']
    mileage_m   = int(top['mileage_m'])
    res         = window.resolution_min

    # profile schema：(vdid, direction, mileage_m, day_class, peak_class,
    #                  minute_bucket, mean_speed_x10, ...)
    # 對每個 time_offset 計算對應的 (day_class, peak_class, minute_bucket)。
    offsets = window.time_offsets_min.astype(int)
    ts      = report_time + pd.to_timedelta(offsets, unit='m')
    tod     = (ts.hour * 60 + ts.minute).astype(int)
    minute_bucket = (tod // res) * res
    match_df = pd.DataFrame({
        'time_offset_min': offsets.astype('int32'),
        'minute_bucket':   minute_bucket.astype('int16'),
        'day_class':       [day_class(t) for t in ts],
        'peak_class':      [_peak_class(int(m)) for m in tod],
    })

    # 同方向、空間範圍篩選
    if direction in ('S', 'E'):
        m_lo = mileage_m - window.upstream_km   * 1000
        m_hi = mileage_m + window.downstream_km * 1000
    else:
        m_lo = mileage_m - window.downstream_km * 1000
        m_hi = mileage_m + window.upstream_km   * 1000
    prof = profile_df[
        (profile_df['direction'] == direction)
        & (profile_df['mileage_m'] >= m_lo)
        & (profile_df['mileage_m'] <= m_hi)
    ].copy()

    # 把每個 profile 列對齊回 time_offset_min
    prof = prof.merge(match_df, on=['day_class', 'peak_class', 'minute_bucket'], how='inner')

    if direction in ('S', 'E'):
        prof['space_offset_m'] = (mileage_m - prof['mileage_m']).astype('int32')
    else:
        prof['space_offset_m'] = (prof['mileage_m'] - mileage_m).astype('int32')

    if prof.empty:
        print('profile 在此事故的時空窗內無匹配列；僅顯示觀測速度熱圖。')
        heatmap(obs_mat, window, top.to_dict(), title='觀測速度熱圖')
        plt.show()
    else:
        hist_mat = rasterize_window(prof, window, value_col='mean_speed_x10')
        diff_mat = obs_mat - hist_mat
        compare_heatmaps(
            [obs_mat, hist_mat, diff_mat],
            ['觀測速度', '歷史基準', '差異（觀測 − 歷史）'],
            window,
            top.to_dict(),
        )
        plt.show()

except FileNotFoundError as e:
    print(f'歷史 profile 尚未建置（{e}），僅顯示觀測速度熱圖。')
    heatmap(obs_mat, window, top.to_dict(), title='觀測速度熱圖')
    plt.show()

## 延伸探索建議

以下查詢可直接在新格中執行：

```python
# VD 布點空間分布（各路線里程範圍）
con.sql("""
    SELECT freeway, direction,
           MIN(mileage_m)/1000.0 AS start_km,
           MAX(mileage_m)/1000.0 AS end_km,
           COUNT(*) AS vd_count
    FROM vd_dim
    GROUP BY freeway, direction
    ORDER BY freeway, direction
""").df()

# 各路線速度中位數（station_5m 全量掃描，需數十秒）
con.sql("""
    SELECT freeway,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY speed_x10 / 10.0) AS median_speed_kmh
    FROM station_5m
    WHERE speed_x10 > 0
    GROUP BY freeway
""").df()

# 低覆蓋率事故的通報時段分析
con.sql("""
    SELECT FLOOR(EXTRACT(HOUR FROM report_time)) AS hour,
           COUNT(*) AS n_accidents,
           AVG(filled_ratio) AS avg_coverage
    FROM accident_samples_metadata_5m
    GROUP BY 1 ORDER BY 1
""").df()
```